<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h1 style="margin-top:0; color:#1d4ed8; font-size:34px;">
⚖️ 05 Model Comparison
</h1>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook consolidates and compares the main modeling stages developed throughout the <b>AI-based early-warning system</b> project.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
Instead of keeping multiple isolated experimental notebooks, this workflow brings the most important modeling approaches into one structured evaluation pipeline.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
🎯 Comparison Scope
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; margin-top:0; color:#1e40af;"><b>This notebook compares:</b></p>

<ul style="font-size:16px; line-height:2; margin-bottom:0;">
  <li>Demographic and registration baseline features</li>
  <li>Early assessment performance indicators</li>
  <li>Early assessment combined with VLE engagement behavior</li>
</ul>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The main objective is to evaluate how prediction performance evolves as more educational signals become available during the course timeline. This helps measure the real contribution of academic performance, student engagement, learning behavior, and timing-related features to early-risk prediction.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
Unlike full-course prediction approaches, the models analyzed here are designed for realistic educational intervention scenarios, where predictions must be generated before the course is completed.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
🧪 Workflow Steps
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; margin-top:0; color:#1e40af;"><b>The workflow includes:</b></p>

<ul style="font-size:16px; line-height:2; margin-bottom:0;">
  <li>Loading and preparing processed datasets</li>
  <li>Comparing multiple feature subsets</li>
  <li>Training machine-learning models</li>
  <li>Evaluating predictive performance</li>
  <li>Analyzing the impact of engagement and assessment signals</li>
  <li>Identifying the strongest early-warning configuration</li>
</ul>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
By the end of this notebook, the project provides a clear understanding of which modeling stage offers the best balance between early prediction capability and overall predictive performance within online learning environments.
</p>

</div>

<h2 style="color:#EA580C;">📦 Imports and Configuration</h2>

This section imports the required Python libraries and defines the main configuration values used throughout the model comparison notebook.

The imported libraries cover data handling, preprocessing, model training, evaluation metrics, and machine-learning algorithms.

In [1]:
# Import libraries for data manipulation.
import pandas as pd
import numpy as np

# Import Path to handle file and folder paths.
from pathlib import Path

# Import tools for splitting the dataset into training and testing sets.
from sklearn.model_selection import train_test_split

# Import preprocessing tools for numerical and categorical features.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Pipeline to combine preprocessing and model training steps.
from sklearn.pipeline import Pipeline

# Import evaluation metrics for model comparison.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier
import os
import joblib
# Import machine-learning models.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Try to import XGBoost if it is installed.
# If not available, the notebook will continue without stopping.
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed. The notebook will run Logistic Regression and Random Forest only.")

# Define global random state for reproducible results.
RANDOM_STATE = 42

# Define the percentage of data used for testing.
TEST_SIZE = 0.2

# Define the processed-data folder path.
DATA_DIR = Path("C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\data\\processed")

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The required libraries and configuration values were prepared successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The notebook is now ready to load processed datasets, build preprocessing pipelines, train multiple models, and compare their performance in a reproducible way.
</p>

</div>

<h2 style="color:#EA580C;">📂 Load Processed Datasets</h2>

This section loads the processed datasets produced by the previous notebooks.

The comparison uses three dataset versions: the full-course dataset, the first 25% early-warning dataset, and the first 50% early-warning dataset.

In [2]:
# Load the full processed student-course dataset.
full_df = pd.read_csv(DATA_DIR / "student_course_features.csv")

# Load the first 25% early-warning dataset.
df_25 = pd.read_csv(DATA_DIR / "student_features_first_25.csv")

# Load the first 50% early-warning dataset.
df_50 = pd.read_csv(DATA_DIR / "student_features_first_50.csv")

# Load the Selected_features_Q1 dataset.
df_selected_q1 = pd.read_csv(DATA_DIR / "Selected_features_Q1.csv")

# Load the Selected_features_Q2 dataset.
df_selected_q2 = pd.read_csv(DATA_DIR / "Selected_features_Q2.csv")

# Display the shape of each dataset.
print("Full dataset:", full_df.shape)
print("First 25% dataset:", df_25.shape)
print("First 50% dataset:", df_50.shape)
print("Selected features_Q1 dataset:", df_selected_q1.shape)
print("Selected features_Q2 dataset:", df_selected_q2.shape)

# Preview the full dataset.
full_df.head()

Full dataset: (32480, 60)
First 25% dataset: (32480, 51)
First 50% dataset: (32480, 51)
Selected features_Q1 dataset: (32480, 17)
Selected features_Q2 dataset: (32480, 16)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,avg_submission_delay,avg_assessment_relative_time,avg_submitted_relative_time,avg_assessment_arab_day,avg_submitted_arab_day,avg_submission_delay_relative,avg_submission_delay_arab_days,assessments_first_50,workload_ratio,workload_level
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,-1.8,0.426119,0.419403,63.91791,62.910448,-6.716418e-03,-1.007463e+00,3.0,2.0,very_high
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,0.426119,0.426119,63.91791,63.917910,-1.110223e-17,-1.598721e-15,3.0,0.5,light
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,0.000000,0.000000,0.00000,0.000000,0.000000e+00,0.000000e+00,0.0,0.5,light
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,-2.0,0.426119,0.418657,63.91791,62.798507,-7.462687e-03,-1.119403e+00,3.0,0.5,light
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,11.4,0.426119,0.468657,63.91791,70.298507,4.253731e-02,6.380597e+00,3.0,0.5,light


In [3]:
df_selected_q1.rename(columns={"target": "at_risk"}, inplace=True)
df_selected_q2.rename(columns={"target": "at_risk"}, inplace=True)

#map the values of the at risk column to 0 and 1
df_selected_q1["at_risk"] = df_selected_q1["at_risk"].map({"successful": 0, "at_risk": 1})
df_selected_q2["at_risk"] = df_selected_q2["at_risk"].map({"successful": 0, "at_risk": 1})

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The processed datasets were loaded successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The full dataset contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">60</code> 
columns, while both early-warning datasets contain 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">51</code> 
columns.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This confirms that all datasets share the same student-course record level, while the full dataset includes additional full-course features not available in the early-warning versions.
</p>

</div>

<h2 style="color:#EA580C;">🎯 Target Preparation</h2>

This section converts the existing target column into a binary label for model training.

The value `1` represents at-risk students, while `0` represents successful students.

In [4]:
def prepare_target(df):
    df = df.copy()

    # Make sure the target column exists.
    if "target" not in df.columns:
        raise ValueError("The dataset must contain a target column.")

    # Convert target labels into binary values.
    df["at_risk"] = df["target"].map({
        "at_risk": 1,
        "successful": 0
    })

    # Check for unexpected target values.
    if df["at_risk"].isnull().any():
        print("Unmapped target values:", df.loc[df["at_risk"].isnull(), "target"].unique())
        raise ValueError("Some target values could not be mapped.")

    return df


full_df = prepare_target(full_df)
df_25 = prepare_target(df_25)
df_50 = prepare_target(df_50)

print(f'Target distribution in full dataset:\n{full_df["at_risk"].value_counts()}')
print(f'Normalized target distribution in full dataset:\n{full_df["at_risk"].value_counts(normalize=True) * 100}')

Target distribution in full dataset:
at_risk
1    17118
0    15362
Name: count, dtype: int64
Normalized target distribution in full dataset:
at_risk
1    52.703202
0    47.296798
Name: proportion, dtype: float64


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The target column was converted successfully into a binary classification label.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The dataset contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">17,118</code> 
at-risk records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">15,362</code> 
successful records.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This means that approximately 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">52.7%</code> 
of the records are at risk, while 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">47.3%</code> 
are successful, showing a relatively balanced target distribution.
</p>

</div>

<h2 style="color:#EA580C;">🧩 Define Feature Groups</h2>

This section defines the feature groups used in the model comparison.

The notebook compares three stages: baseline features, first 25% early-warning features, and first 50% early-warning features.

In [5]:
baseline_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "disability",
    "workload_ratio",
    "workload_level"
]

# Keep only available baseline columns.
baseline_features = [col for col in baseline_features if col in full_df.columns]

exclude_cols = {
    "id_student", "final_result", "target",'at_risk',
    "module_presentation_length", "vle_module_presentation_length"
}

# Define feature sets for the early-warning datasets.
features_25 = [col for col in df_25.columns if col not in exclude_cols]
features_50 = [col for col in df_50.columns if col not in exclude_cols]
selected_features_q1 = [col for col in df_selected_q1.columns if col not in exclude_cols]
selected_features_q2 = [col for col in df_selected_q2.columns if col not in exclude_cols]

print("Baseline features:", len(baseline_features))
print(baseline_features)
print("25% features:", len(features_25))
print("25% features list:", features_25)    
print("50% features:", len(features_50))
print("50% features list:", features_50)
print("Selected features Q1:", len(selected_features_q1))
print("Selected features Q1 list:", selected_features_q1)
print("Selected features Q2:", len(selected_features_q2))
print("Selected features Q2 list:", selected_features_q2)

Baseline features: 11
['code_module', 'code_presentation', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'disability', 'workload_ratio', 'workload_level']
25% features: 46
25% features list: ['code_module', 'code_presentation', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'disability', 'active_days_until_cutoff', 'unique_sites_until_cutoff', 'unique_activity_types_until_cutoff', 'clicks_per_active_day_until_cutoff', 'active_days_ratio_until_cutoff', 'arab_active_days_equivalent_until_cutoff', 'dataplus', 'dualpane', 'externalquiz', 'forumng', 'glossary', 'homepage', 'htmlactivity', 'oucollaborate', 'oucontent', 'ouelluminate', 'ouwiki', 'page', 'questionnaire', 'quiz', 'repeatactivity', 'resource', 'sharedsubpage', 'subpage', 'url', 'avg_score_until_cutoff', 'submitted_assessments_until_cutoff', 'late_submissions_until_cutoff', 'avg_submission_delay_until_cutoff', 'avg_assessment_relative_time_until_

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Three feature groups were defined successfully for model comparison.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The baseline model uses 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">11</code> 
demographic and registration-related features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Both the 25% and 50% early-warning datasets contain 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">46</code> 
usable predictive features after excluding identifiers, target columns, and timeline-length columns.
</p>

</div>

<h2 style="color:#EA580C;">🛠️ Build Reusable Training Function</h2>

This section defines reusable helper functions for preprocessing, training, and evaluating machine-learning models.

These functions make the comparison workflow cleaner by applying the same data-splitting, preprocessing, training, and evaluation logic across all model stages.

In [6]:
def split_feature_types(df, features):
    # Separate categorical and numerical features.
    categorical_features = [
        col for col in features
        if df[col].dtype == "object" or str(df[col].dtype).startswith("category")
    ]

    numeric_features = [
        col for col in features
        if col not in categorical_features
    ]

    return categorical_features, numeric_features


def build_preprocessor(df, features):
    categorical_features, numeric_features = split_feature_types(df, features)

    # Apply encoding for categorical features and scaling for numerical features.
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", StandardScaler(), numeric_features)
        ]
    )


def evaluate_predictions(y_test, y_pred):
    # Return the main evaluation metrics.
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_at_risk": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_at_risk": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_at_risk": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    }


def train_and_evaluate(df, features, model_name, model, save_model=True):
    df = df.copy()

    # Handle missing and infinite values.
    categorical_features, numeric_features = split_feature_types(df, features)
    df[categorical_features] = df[categorical_features].fillna("Unknown")
    df[numeric_features] = df[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)

    X = df[features]
    y = df["at_risk"]

    # Split data into train and test sets.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    # Build full preprocessing + model pipeline.
    pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(df, features)),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = evaluate_predictions(y_test, y_pred)
    metrics["model"] = model_name
    metrics["n_features"] = len(features)

    print(f"=== {model_name} ===")
    print("Accuracy:", metrics["accuracy"])
    print("Classification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Save model pipeline for production
    if save_model:
        os.makedirs("models", exist_ok=True)

        safe_model_name = (
            model_name
            .replace(" ", "_")
            .replace("%", "percent")
            .replace("-", "_")
            .replace("/", "_")
        )

        model_path = f"models/{safe_model_name}.pkl"
        joblib.dump(pipeline, model_path)

        print(f"Saved model to: {model_path}")

    return metrics, pipeline

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Reusable training and evaluation functions were created successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
These functions standardize preprocessing, train-test splitting, model fitting, and metric calculation across all model comparison stages.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This makes the results more consistent, easier to reproduce, and easier to compare fairly.
</p>

</div>

<h2 style="color:#EA580C;">📊 Train Baseline Models</h2>

This section trains baseline models using only demographic and registration-related features.

This stage replaces the core purpose of the archived `V01 Baseline.ipynb` notebook and provides the first performance reference point for comparison.

In [7]:
experiments = []
trained_models = {}

baseline_models = {
    "Baseline - Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "Baseline - Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
}

# Add XGBoost only if it is installed.
if XGBOOST_AVAILABLE:
    baseline_models["Baseline - XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )

# Train and evaluate each baseline model.
for name, model in baseline_models.items():
    metrics, fitted_model = train_and_evaluate(
        full_df,
        baseline_features,
        name,
        model,
        save_model=False
    )

    experiments.append(metrics)
    trained_models[name] = fitted_model


# Select and save only the best baseline model.
best_baseline_metrics = max(
    [m for m in experiments if m["model"].startswith("Baseline - ")],
    key=lambda x: x["f1_at_risk"]
)
'''
best_baseline_name = best_baseline_metrics["model"]
best_baseline_model = trained_models[best_baseline_name]

os.makedirs("models", exist_ok=True)

safe_model_name = (
    best_baseline_name
    .replace(" ", "_")
    .replace("%", "percent")
    .replace("-", "_")
    .replace("/", "_")
)

model_path = f"models/{safe_model_name}.pkl"
joblib.dump(best_baseline_model, model_path)

print(f"Saved best baseline model to: {model_path}")'''

=== Baseline - Logistic Regression ===
Accuracy: 0.625
Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.64      0.62      3072
           1       0.65      0.61      0.63      3424

    accuracy                           0.62      6496
   macro avg       0.63      0.63      0.62      6496
weighted avg       0.63      0.62      0.63      6496

Confusion Matrix:
[[1962 1110]
 [1326 2098]]
=== Baseline - Random Forest ===
Accuracy: 0.584051724137931
Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.54      0.55      3072
           1       0.60      0.62      0.61      3424

    accuracy                           0.58      6496
   macro avg       0.58      0.58      0.58      6496
weighted avg       0.58      0.58      0.58      6496

Confusion Matrix:
[[1667 1405]
 [1297 2127]]
=== Baseline - XGBoost ===
Accuracy: 0.6297721674876847
Classification Report:
              p

'\nbest_baseline_name = best_baseline_metrics["model"]\nbest_baseline_model = trained_models[best_baseline_name]\n\nos.makedirs("models", exist_ok=True)\n\nsafe_model_name = (\n    best_baseline_name\n    .replace(" ", "_")\n    .replace("%", "percent")\n    .replace("-", "_")\n    .replace("/", "_")\n)\n\nmodel_path = f"models/{safe_model_name}.pkl"\njoblib.dump(best_baseline_model, model_path)\n\nprint(f"Saved best baseline model to: {model_path}")'

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The baseline models were trained and evaluated successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Logistic Regression achieved an accuracy of 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">62.5%</code>, 
which is better than Random Forest at this stage.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Since this baseline uses only demographic and registration-related features, the results show that these features provide a useful but limited prediction signal.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The next stages are expected to improve performance by adding assessment and VLE engagement features.
</p>

</div>

<h2 style="color:#EA580C;">⏱️ Train Early-Warning Models — First 25%</h2>

This section trains early-warning models using only the features available during the first 25% of the course timeline.

This stage evaluates how well the system can detect at-risk students very early in the learning process.

In [8]:
models_25 = {
    "25% Early Warning - Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "25% Early Warning - Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
}

# Add XGBoost only if available.
if XGBOOST_AVAILABLE:
    models_25["25% Early Warning - XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )

# Train and evaluate each 25% early-warning model.
for name, model in models_25.items():
    metrics, fitted_model = train_and_evaluate(
        df_25,
        features_25,
        name,
        model,
        save_model=False
    )

    experiments.append(metrics)
    trained_models[name] = fitted_model


# Select and save only the best 25% early-warning model.
best_25_metrics = max(
    [m for m in experiments if m["model"].startswith("25% Early Warning - ")],
    key=lambda x: x["f1_at_risk"]
)
'''
best_25_name = best_25_metrics["model"]
best_25_model = trained_models[best_25_name]

os.makedirs("models", exist_ok=True)

safe_model_name = (
    best_25_name
    .replace(" ", "_")
    .replace("%", "percent")
    .replace("-", "_")
    .replace("/", "_")
)

model_path = f"models/{safe_model_name}.pkl"
joblib.dump(best_25_model, model_path)

print(f"Saved best 25% early-warning model to: {model_path}")'''

=== 25% Early Warning - Logistic Regression ===
Accuracy: 0.8017241379310345
Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.87      0.81      3072
           1       0.86      0.74      0.80      3424

    accuracy                           0.80      6496
   macro avg       0.81      0.81      0.80      6496
weighted avg       0.81      0.80      0.80      6496

Confusion Matrix:
[[2667  405]
 [ 883 2541]]
=== 25% Early Warning - Random Forest ===
Accuracy: 0.8069581280788177
Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.86      0.81      3072
           1       0.86      0.76      0.81      3424

    accuracy                           0.81      6496
   macro avg       0.81      0.81      0.81      6496
weighted avg       0.81      0.81      0.81      6496

Confusion Matrix:
[[2650  422]
 [ 832 2592]]
=== 25% Early Warning - XGBoost ===
Accuracy: 0.81357758620689

'\nbest_25_name = best_25_metrics["model"]\nbest_25_model = trained_models[best_25_name]\n\nos.makedirs("models", exist_ok=True)\n\nsafe_model_name = (\n    best_25_name\n    .replace(" ", "_")\n    .replace("%", "percent")\n    .replace("-", "_")\n    .replace("/", "_")\n)\n\nmodel_path = f"models/{safe_model_name}.pkl"\njoblib.dump(best_25_model, model_path)\n\nprint(f"Saved best 25% early-warning model to: {model_path}")'

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The 25% early-warning models were trained and evaluated successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Both Logistic Regression and Random Forest achieved around 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">80%</code> 
accuracy, which is a clear improvement over the baseline models.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
This shows that assessment and engagement features available in the first quarter of the course provide much stronger predictive signals than demographic and registration data alone.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Random Forest performed slightly better, reaching an accuracy of approximately 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">80.7%</code>.
</p>

</div>

<h2 style="color:#EA580C;">⏱️ Train Early-Warning Models — First 50%</h2>

This section trains early-warning models using the features available during the first 50% of the course timeline.

This stage evaluates how model performance improves when more mid-course learning behavior and assessment signals become available.

In [9]:
models_50 = {
    "50% Early Warning - Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "50% Early Warning - Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
}

# Add XGBoost only if available.
if XGBOOST_AVAILABLE:
    models_50["50% Early Warning - XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )

# Train and evaluate each 50% early-warning model.
for name, model in models_50.items():
    metrics, fitted_model = train_and_evaluate(
        df_50,
        features_50,
        name,
        model,
        save_model=False
    )

    experiments.append(metrics)
    trained_models[name] = fitted_model

'''
# Select and save only the best 50% early-warning model.
best_50_metrics = max(
    [m for m in experiments if m["model"].startswith("50% Early Warning - ")],
    key=lambda x: x["f1_at_risk"]
)

best_50_name = best_50_metrics["model"]
best_50_model = trained_models[best_50_name]'''

=== 50% Early Warning - Logistic Regression ===
Accuracy: 0.8528325123152709
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.90      0.85      3072
           1       0.90      0.81      0.85      3424

    accuracy                           0.85      6496
   macro avg       0.86      0.86      0.85      6496
weighted avg       0.86      0.85      0.85      6496

Confusion Matrix:
[[2780  292]
 [ 664 2760]]
=== 50% Early Warning - Random Forest ===
Accuracy: 0.8634544334975369
Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.92      0.86      3072
           1       0.92      0.82      0.86      3424

    accuracy                           0.86      6496
   macro avg       0.87      0.87      0.86      6496
weighted avg       0.87      0.86      0.86      6496

Confusion Matrix:
[[2815  257]
 [ 630 2794]]
=== 50% Early Warning - XGBoost ===
Accuracy: 0.86514778325123

'\n# Select and save only the best 50% early-warning model.\nbest_50_metrics = max(\n    [m for m in experiments if m["model"].startswith("50% Early Warning - ")],\n    key=lambda x: x["f1_at_risk"]\n)\n\nbest_50_name = best_50_metrics["model"]\nbest_50_model = trained_models[best_50_name]'

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The 50% early-warning models were trained and evaluated successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Both models achieved stronger results than the baseline and the 25% early-warning stage.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Logistic Regression reached an accuracy of approximately 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">85.3%</code>, 
while Random Forest achieved the best result with approximately 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">86.3%</code> 
accuracy.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This confirms that mid-course assessment and VLE engagement signals provide a richer and more reliable representation of student risk.
</p>

</div>

<h2 style="color:#EA580C;">⏱️ Train Early-Warning Selected features Models — First 25% (Q1)</h2>

This section trains early-warning models using only the Selected features available during the first 25% of the course timeline.

This stage evaluates how well the system can detect at-risk students very early in the learning process.

In [10]:
models_selected_25 = {
    "selected features of the 25% Early Warning - Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "selected features of the 25% Early Warning - Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
}

# Add XGBoost only if available.
if XGBOOST_AVAILABLE:
    models_selected_25["selected features of the 25% Early Warning - XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )

# Train and evaluate each selected features 25% early-warning model.
for name, model in models_selected_25.items():
    metrics, fitted_model = train_and_evaluate(
        df_selected_q1,
        selected_features_q1,
        name,
        model,
        save_model=False
    )

    experiments.append(metrics)
    trained_models[name] = fitted_model


# Select and save only the best selected features 25% early-warning model.
best_selected_25_metrics = max(
    [m for m in experiments if m["model"].startswith("selected features of the 25% Early Warning - ")],
    key=lambda x: x["f1_at_risk"]
)

best_selected_25_name = best_selected_25_metrics["model"]
best_selected_25_model = trained_models[best_selected_25_name]

os.makedirs("models", exist_ok=True)

safe_model_name = (
    best_selected_25_name
    .replace(" ", "_")
    .replace("%", "percent")
    .replace("-", "_")
    .replace("/", "_")
)

model_path = f"models/{safe_model_name}.pkl"
joblib.dump(best_selected_25_model, model_path)

print(f"Saved best selected features 25% early-warning model to: {model_path}")

=== selected features of the 25% Early Warning - Logistic Regression ===
Accuracy: 0.7572352216748769
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.84      0.77      3072
           1       0.83      0.68      0.75      3424

    accuracy                           0.76      6496
   macro avg       0.77      0.76      0.76      6496
weighted avg       0.77      0.76      0.76      6496

Confusion Matrix:
[[2585  487]
 [1090 2334]]
=== selected features of the 25% Early Warning - Random Forest ===
Accuracy: 0.7994150246305419
Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.84      0.80      3072
           1       0.84      0.76      0.80      3424

    accuracy                           0.80      6496
   macro avg       0.80      0.80      0.80      6496
weighted avg       0.80      0.80      0.80      6496

Confusion Matrix:
[[2590  482]
 [ 821 2603]]
=== selected

<h2 style="color:#EA580C;">⏱️ Train Early-Warning Selected features Models — First 50% (Q2)</h2>

This section trains early-warning models using the features available during the first 50% of the course timeline.

This stage evaluates how model performance improves when more mid-course learning behavior and assessment signals become available.

In [11]:
models_selected_50 = {
    "selected features of the 50% Early Warning - Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "selected features of the 50% Early Warning - Random Forest": RandomForestClassifier(
        n_estimators=700,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
}

# Add XGBoost only if available.
if XGBOOST_AVAILABLE:
    models_selected_50["selected features of the 50% Early Warning - XGBoost"] = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE
    )

# Train and evaluate each selected features 50% early-warning model.
for name, model in models_selected_50.items():
    metrics, fitted_model = train_and_evaluate(
        df_selected_q2,
        selected_features_q2,
        name,
        model,
        save_model=False
    )

    experiments.append(metrics)
    trained_models[name] = fitted_model


# Select and save only the best selected features 50% early-warning model.
best_selected_50_metrics = max(
    [m for m in experiments if m["model"].startswith("selected features of the 50% Early Warning - ")],
    key=lambda x: x["f1_at_risk"]
)

best_selected_50_name = best_selected_50_metrics["model"]
best_selected_50_model = trained_models[best_selected_50_name]

os.makedirs("models", exist_ok=True)

safe_model_name = (
    best_selected_50_name
    .replace(" ", "_")
    .replace("%", "percent")
    .replace("-", "_")
    .replace("/", "_")
)

model_path = f"models/{safe_model_name}.pkl"
joblib.dump(best_selected_50_model, model_path)

print(f"Saved best selected features 50% early-warning model to: {model_path}")

=== selected features of the 50% Early Warning - Logistic Regression ===
Accuracy: 0.8132697044334976
Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.86      0.81      3072
           1       0.86      0.77      0.81      3424

    accuracy                           0.81      6496
   macro avg       0.82      0.82      0.81      6496
weighted avg       0.82      0.81      0.81      6496

Confusion Matrix:
[[2650  422]
 [ 791 2633]]
=== selected features of the 50% Early Warning - Random Forest ===
Accuracy: 0.8516009852216748
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.90      0.85      3072
           1       0.90      0.81      0.85      3424

    accuracy                           0.85      6496
   macro avg       0.85      0.85      0.85      6496
weighted avg       0.86      0.85      0.85      6496

Confusion Matrix:
[[2753  319]
 [ 645 2779]]
=== selected

<h2 style="color:#EA580C;">📋 Final Model Comparison Table</h2>

This section summarizes all experiment results in one comparison table.

The models are sorted by F1-score for the at-risk class to identify the strongest model for early-risk prediction.

In [12]:
results_df = pd.DataFrame(experiments)

results_df["group"] = results_df["model"].str.replace(
    r" - (Logistic Regression|Random Forest|XGBoost)$",
    "",
    regex=True
)

results_df = (
    results_df
    .sort_values(by="f1_at_risk", ascending=False)
    .groupby("group", as_index=False)
    .first()
)

results_df = results_df[[
    "model",
    "n_features",
    "accuracy",
    "precision_at_risk",
    "recall_at_risk",
    "f1_at_risk"
]].sort_values(by="f1_at_risk", ascending=False)

# Round metric values for clearer display.
for col in ["accuracy", "precision_at_risk", "recall_at_risk", "f1_at_risk"]:
    results_df[col] = results_df[col].round(4)

display(results_df)

,model,n_features,accuracy,precision_at_risk,recall_at_risk,f1_at_risk
1,50% Early Warning - XGBoost,46,0.8651,0.9139,0.8216,0.8653
4,selected features of the 50% Early Warning - R...,15,0.8516,0.8970,0.8116,0.8522
0,25% Early Warning - XGBoost,46,0.8136,0.8636,0.7675,0.8127
3,selected features of the 25% Early Warning - R...,16,0.7994,0.8438,0.7602,0.7998
2,Baseline - XGBoost,11,0.6298,0.6424,0.6714,0.6566


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final comparison table shows that the 50% early-warning models achieved the strongest overall performance.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The best model was 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">50% Early Warning - XGBoost</code>, 
achieving an F1-score of 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">0.8653</code> 
for the at-risk class.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This confirms that using mid-course assessment and VLE engagement features provides the most effective prediction signal for identifying at-risk students.
</p>

</div>

<h2 style="color:#EA580C;">💾 Save Comparison Results</h2>

This section saves the final model comparison results for reporting and future analysis.

In [13]:
OUTPUT_DIR = Path("../reports")

# Create reports folder if it does not exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save only the best model from each experiment group.
results_df.to_csv(
    OUTPUT_DIR / "model_comparison_results.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "model_comparison_results.csv")

Saved: ..\reports\model_comparison_results.csv


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The model comparison results were saved successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The exported CSV file can be used later for reporting, visualization, and performance analysis.
</p>

</div>

<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h2 style="margin-top:0; color:#1d4ed8; font-size:30px;">
🧠 Interpretation Notes
</h2>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This section summarizes the main findings from the model comparison experiments.
</p>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<ul style="font-size:16px; line-height:2; margin:0;">
  <li>The <b>baseline models</b> achieved the weakest performance because they relied only on demographic and registration-related features.</li>
  <li>The <b>25% early-warning models</b> showed a major improvement after adding early assessment and engagement behavior features.</li>
  <li>The <b>50% early-warning models</b> achieved the best overall performance because they captured richer mid-course behavioral and academic signals.</li>
  <li>The results confirm that student engagement behavior and assessment activity provide strong predictive value for early-risk detection.</li>
  <li>Although the 50% models achieved the highest accuracy and F1-score, the <b>25% models remain highly valuable</b> because they can detect risk earlier in the course timeline.</li>
  <li>For practical early-warning systems, <b>recall and F1-score for the at-risk class</b> are more important than accuracy alone, since missing at-risk students may reduce the effectiveness of educational intervention.</li>
</ul>

</div>

</div>

<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h2 style="margin-top:0; color:#1d4ed8; font-size:30px;">
✅ Notebook Summary
</h2>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook successfully compared the main modeling stages developed throughout the <b>AI-based early-warning system</b> project using a unified and reproducible machine-learning workflow.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The experiments demonstrated how predictive performance improves as additional educational signals become available during the course timeline. The baseline models provided the weakest results because they relied mainly on demographic and registration-related information, while the early-warning models achieved significantly stronger performance after incorporating assessment behavior and VLE engagement features.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
📌 Key Findings
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<ul style="font-size:16px; line-height:2; margin:0;">
  <li>The <b>25% early-warning stage</b> showed that meaningful student-risk prediction is possible even during the early part of the course.</li>
  <li>The <b>50% early-warning stage</b> achieved the strongest overall results by using richer mid-course behavioral and academic signals.</li>
  <li>Student engagement activity, assessment performance, and timing-related behavior provide strong predictive value for identifying at-risk students.</li>
</ul>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
Overall, this notebook transformed multiple isolated experiments into a clean comparison framework that supports fair evaluation, reproducibility, and clearer interpretation of model performance across different early-warning stages.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The final comparison results provide a strong experimental foundation for future deployment stages, intelligent intervention systems, and personalized educational support mechanisms within AI-driven learning environments.
</p>

</div>